# CAM → ICFR → PCAOB — Google Colab research harness
GitHub-hosted reproducible synthetic smoke test. Jev/OpenJev is **not installed**; no real Drive data are transferred to GitHub. Run cells in Colab after opening this notebook from GitHub.


In [ ]:
!git clone --quiet --branch research/jev-ai-audit-proposal https://github.com/Saehon/Saeid-Homayoun.git /content/jev_audit
!python /content/jev_audit/research/jev-ai-audit/three_stage_harness.py


## Optional private Google Drive mount
Authenticate interactively. Existing canonical CAM and ICFR sheets stay in Drive. Never commit raw data or secrets. Actual model scoring is a later gated step.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted; no private dataset was uploaded to GitHub.')


## Planned next step
Verify CAM outcome labels, ICFR effective/ineffective/unknown taxonomy, industry/domain mapping, chronological holdouts, and Jev/OpenJev model license and inference API. PCAOB output remains inspection triage with human approval only. Google Cloud/Vertex AI and Google AI services require separate project access, API enablement and potentially billing; this notebook does not activate them.


## Unified model registry: Jev candidates, FinBERT, Granite, Claude, Google AI, baseline
This is an integration **manifest**, not a claim of installed or executed third-party models. RAD is the development methodology; MCP/A2A/Agent Skills are orchestration mechanisms; MLflow tracks experiments. Start with a real executable sklearn baseline on **synthetic** CAM-like observations. All external models require verified licenses, endpoint access, privacy clearance and a prediction adapter before benchmarking.

In [ ]:
MODEL_REGISTRY = {
    "sklearn_logistic_baseline": {"kind": "local", "status": "executable_synthetic"},
    "finbert": {"kind": "hugging_face", "status": "adapter_pending"},
    "ibm_granite": {"kind": "hugging_face", "status": "adapter_pending"},
    "openjev_rlcd": {"upstream": "https://github.com/ZimmyGao/openjev-rlcd", "status": "adapter_pending"},
    "openjev_verdict": {"upstream": "https://github.com/Heman10x-NGU/openJev-verdict-2.0", "status": "adapter_pending"},
    "openjevx": {"upstream": "https://github.com/muthuishere/openjevx", "status": "adapter_pending"},
    "claude_agent": {"kind": "external_api", "status": "credentials_and_privacy_review_pending"},
    "google_gemini_vertex": {"kind": "google_cloud", "status": "project_and_billing_review_pending"},
}
for model, details in MODEL_REGISTRY.items():
    print(model, details["status"])

## Executable synthetic CAM pilot with MLflow
This is a **technical pipeline test**, not research evidence. It trains logistic regression on generated data, logs a Brier score and dataset type to a temporary local MLflow store. The store disappears when Colab resets. For durable runs, use an approved private Drive-backed tracking directory or tracking server; never commit experiment stores to public GitHub.

In [ ]:
%pip -q install mlflow scikit-learn pandas numpy
import numpy as np
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, average_precision_score
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(20261015)
n = 240
risk = rng.normal(size=n)
industry = rng.integers(0, 3, n)
domain = rng.integers(0, 3, n)
prob = 1/(1+np.exp(-(-0.7+0.8*risk+0.2*(industry==1)+0.3*(domain==2))))
y = rng.binomial(1, prob)
X = np.column_stack([risk, industry==1, industry==2, domain==1, domain==2])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.3, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000).fit(Xtr,ytr)
p = model.predict_proba(Xte)[:,1]
mlflow.set_tracking_uri("file:///content/mlruns_jev_cam_temporary")
mlflow.set_experiment("synthetic_cam_pipeline_only")
with mlflow.start_run(run_name="rad_cam_baseline_smoke"):
    mlflow.log_params({"model":"logistic_regression","data":"SYNTHETIC","stage":"CAM","seed":20261015})
    mlflow.log_metric("brier",float(brier_score_loss(yte,p)))
    mlflow.log_metric("pr_auc",float(average_precision_score(yte,p)))
print("SYNTHETIC ONLY | Brier:",round(brier_score_loss(yte,p),4),"PR-AUC:",round(average_precision_score(yte,p),4))
print("No real CAM labels and no Jev/FinBERT/Granite/Claude/Gemini inference.")

## Optional durable MLflow archive in canonical Google Drive
Run after mounting Drive. This uses a single MLflow tracking directory in Drive; it does not create a new spreadsheet. **Opt-in** and inspect permissions before enabling. To avoid duplicates, reuse the same directory for later experiments.

In [ ]:
SAVE_MLFLOW_TO_DRIVE = False
if SAVE_MLFLOW_TO_DRIVE:
    from pathlib import Path
    tracking = Path("/content/drive/MyDrive/Research/jev-ai-audit/mlruns")
    tracking.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(tracking.as_uri())
    print("Persistent private tracking:", tracking)
else:
    print("Persistent MLflow tracking not enabled.")

## Scientific and security gates
1. CAM first: verified firm-year-topic schema and **independent** outcome labels; CAM presence ≠ CAM effectiveness.
2. ICFR next: effective/ineffective/unknown by industry and accounting/control domain; ASC 606 for US revenue.
3. PCAOB last: inspection triage, immutable evidence lineage and human approval; not an official inspection finding.
4. Third-party inference is **disabled by default**. No Google Cloud or Claude API deployment is made by this notebook. A2A/MCP/Skills/RAD describe future orchestration and iteration, not installed services.
5. Preserve existing canonical Google Sheets; append a stage tab only after validated real outcomes and approval.